# Setup

In [1]:
%pip install -q torch transformers bm25s scikit-learn pandas numpy matplotlib sentence-transformers
%pip install -U datasets accelerate

import csv
import json
import random
from pathlib import Path
import numpy as np
import torch

import bm25s
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import InputExample, SentenceTransformer, losses

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_110722/105137681.py:14: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import InputExample, SentenceTransformer, losses


In [2]:
SEED = 42
DATA_DIR = Path(".")
SUBMISSION_PATH = Path("2022320319_final.csv")
TOP_K = 10

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
rng = random.Random(SEED)

def load_json(path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

catalog = load_json(DATA_DIR / "catalog.json")
test_queries = load_json(DATA_DIR / "test_queries.json")
train_gold = load_json(DATA_DIR / "train_gold.json")

catalog_item_ids = [item["item_id"] for item in catalog]
catalog_id_set = set(catalog_item_ids)
test_instance_ids = [instance["query_id"] for instance in test_queries]
train_instance_ids = [instance["query_id"] for instance in train_gold]

assert len(catalog_item_ids) == 50_000
assert len(set(catalog_item_ids)) == len(catalog_item_ids)
assert len(test_queries) == 19_858
assert len(set(test_instance_ids)) == len(test_instance_ids)
assert all(len(instance["reviews"]) == 5 for instance in test_queries)
assert len(train_gold) == 20_000
assert len(set(train_instance_ids)) == len(train_instance_ids)
assert all(len(instance["reviews"]) == 5 for instance in train_gold)

In [3]:
def recall_at_k_single(ranked_pids, relevant_pids):
    rel_set = set(relevant_pids)
    if not rel_set:
        return None
    hit = sum(1 for pid in ranked_pids[:TOP_K] if pid in rel_set)
    return hit / len(rel_set)


def evaluate_recall(run, qrels):
    out = {}
    rec_vals = []
    for qid, ranked in tqdm(
        run.items(), desc=f"@k={TOP_K}", leave=False
    ):
        rel_pids = qrels.get(qid, set())
        r = recall_at_k_single(ranked, rel_pids)
        if r is not None:
            rec_vals.append(r)

    out[f"Recall@{TOP_K}"] = (
        round(float(np.mean(rec_vals)), 4) if rec_vals else 0.0
    )

    out["n_queries"] = len(run)
    return out
    
def get_item_text(item: dict) -> str:
    return " ".join(
        str(item[k])
        for k in ("title", "name", "description", "text")
        if item.get(k)
    )

def get_query_text(instance: dict) -> str:
    parts = []

    for rev in instance["reviews"]:
        if rev["text"]:
            parts.append(str(rev["text"]))

    text = " ".join(parts)
    return text

In [4]:
corpus_tokens = bm25s.tokenize(
    [get_item_text(item) for item in catalog], stopwords="en"
)
retriever = bm25s.BM25(k1=1.5, b=0.75)
retriever.index(corpus_tokens)

train_query_tokens = bm25s.tokenize(
    [get_query_text(inst) for inst in train_gold], stopwords="en"
)
train_results, _ = retriever.retrieve(
    train_query_tokens, corpus=catalog_item_ids, k=TOP_K
)

train_preds = {
    inst["query_id"]: list(train_results[i])
    for i, inst in enumerate(train_gold)
}
qrels_train = {inst["query_id"]: [inst["item_id"]] for inst in train_gold}

metrics = evaluate_recall(train_preds, qrels_train)
print(metrics)

test_query_tokens = bm25s.tokenize(
    [get_query_text(inst) for inst in test_queries], stopwords="en"
)
test_results, _ = retriever.retrieve(
    test_query_tokens, corpus=catalog_item_ids, k=TOP_K
)

test_preds = {
    inst["query_id"]: list(test_results[i])
    for i, inst in enumerate(test_queries)
}

{'Recall@10': 0.277, 'n_queries': 20000}


In [5]:
# 2. Bi-Encoder 로드 (MS MARCO 학습 모델 - Amazon 데이터 미포함)
MODEL_NAME = "sentence-transformers/msmarco-distilbert-base-v4"
model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = 512

catalog_texts = [get_item_text(item) for item in catalog]
doc_embs = model.encode(
    catalog_texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True,
)

def retrieve_dense(queries, top_k=TOP_K, batch_size=256) -> dict[str, list[str]]:
    query_texts = [get_query_text(inst) for inst in queries]
    query_embs = model.encode(
        query_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    preds = {}
    # 20,000 x 50,000 전체 행렬 한 번에 계산 시 OOM 방지를 위한 청크 분할
    for start in range(0, len(queries), 1000):
        end = min(start + 1000, len(queries))
        scores = torch.matmul(query_embs[start:end], doc_embs.T)
        top_indices = torch.topk(scores, k=top_k, dim=1).indices.cpu().numpy()

        for i, idx_list in enumerate(top_indices):
            qid = queries[start + i]["query_id"]
            preds[qid] = [catalog_item_ids[idx] for idx in idx_list]

    return preds


# 4. train_gold 평가
dense_train_preds = retrieve_dense(train_gold, top_k=TOP_K)
qrels_train = {inst["query_id"]: [inst["item_id"]] for inst in train_gold}

metrics_dense = evaluate_recall(dense_train_preds, qrels_train)
print(metrics_dense)

# 5. test_queries 예측 생성
dense_test_preds = retrieve_dense(test_queries, top_k=TOP_K)

Batches: 100%|██████████| 79/79 [01:46<00:00,  1.35s/it]
                                                

{'Recall@10': 0.2258, 'n_queries': 20000}


Batches: 100%|██████████| 78/78 [01:46<00:00,  1.37s/it]


In [6]:
# 2. Train / Validation 분할 (과적합 착시 방지용 18,000 : 2,000)
indices = list(range(len(train_gold)))
random.shuffle(indices)
val_size = 2000
val_split = [train_gold[i] for i in indices[:val_size]]
train_split = [train_gold[i] for i in indices[val_size:]]

# item_id -> 상품 텍스트 매핑
id2text = {
    item["item_id"]: get_item_text(item)
    for item in catalog
}

# 3. BM25 Hard Negative 마이닝 (앞서 만든 bm25s retriever 활용)
train_q_tokens = bm25s.tokenize(
    [get_query_text(inst) for inst in train_split], stopwords="en"
)
bm25_top_hits, _ = retriever.retrieve(
    train_q_tokens, corpus=catalog_item_ids, k=10
)

train_examples = []
for i, inst in enumerate(train_split):
    pos_id = inst["item_id"]
    if pos_id not in id2text:
        continue

    # BM25 상위 결과 중 정답이 아닌 첫 번째 상품을 Hard Negative로 선택
    neg_id = next((pid for pid in bm25_top_hits[i] if pid != pos_id), None)
    q_text = get_query_text(inst)
    pos_text = id2text[pos_id]

    if neg_id and neg_id in id2text:
        train_examples.append(InputExample(texts=[q_text, pos_text, id2text[neg_id]]))
    else:
        train_examples.append(InputExample(texts=[q_text, pos_text]))

# 4. Bi-Encoder 파인튜닝 (Listwise Contrastive / Ranking Loss)
MODEL_NAME = "sentence-transformers/msmarco-distilbert-base-v4"
ft_model = SentenceTransformer(MODEL_NAME, device=device)
ft_model.max_seq_length = 384

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=64)
train_loss = losses.MultipleNegativesRankingLoss(ft_model)

EPOCHS = 2
warmup_steps = int(len(train_dataloader) * EPOCHS * 0.1)

ft_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    use_amp=torch.cuda.is_available(),
    show_progress_bar=True,
)

# 5. 파인튜닝 모델 임베딩 및 검색 함수
catalog_texts = [get_item_text(item) for item in catalog]
ft_doc_embs = ft_model.encode(
    catalog_texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_tensor=True,
    normalize_embeddings=True,
)


def retrieve_ft(queries, top_k=TOP_K, batch_size=256) -> dict[str, list[str]]:
    query_texts = [get_query_text(inst) for inst in queries]
    query_embs = ft_model.encode(
        query_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_tensor=True,
        normalize_embeddings=True,
    )

    preds = {}
    for start in range(0, len(queries), 1000):
        end = min(start + 1000, len(queries))
        scores = torch.matmul(query_embs[start:end], ft_doc_embs.T)
        top_indices = torch.topk(scores, k=top_k, dim=1).indices.cpu().numpy()

        for i, idx_list in enumerate(top_indices):
            qid = queries[start + i]["query_id"]
            preds[qid] = [catalog_item_ids[idx] for idx in idx_list]

    return preds


# 6. Validation 평가 및 Test 예측 생성
ft_val_preds = retrieve_ft(val_split, top_k=TOP_K)
qrels_val = {inst["query_id"]: [inst["item_id"]] for inst in val_split}

metrics_ft = evaluate_recall(ft_val_preds, qrels_val)
print("Fine-tuned Validation:", metrics_ft)

ft_test_preds = retrieve_ft(test_queries, top_k=TOP_K)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6485.40it/s]  
                                                                     

Step,Training Loss
500,0.753950


Batches: 100%|██████████| 8/8 [00:02<00:00,  3.02it/s]
                                               

Fine-tuned Validation: {'Recall@10': 0.446, 'n_queries': 2000}


Batches: 100%|██████████| 78/78 [00:26<00:00,  2.99it/s]


# Submission

In [14]:
fieldnames = ["query_id"] + [
    f"item_id_{rank}" for rank in range(1, TOP_K + 1)
]

with SUBMISSION_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

    for instance in test_queries:
        ranked_items = ft_test_preds[instance["query_id"]]
        row = {"query_id": instance["query_id"]}
        row.update({
            f"item_id_{rank}": item_id
            for rank, item_id in enumerate(ranked_items, start=1)
        })
        writer.writerow(row)

with SUBMISSION_PATH.open("r", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    generated_rows = list(reader)

assert reader.fieldnames == fieldnames
assert len(generated_rows) == len(test_queries)
assert [row["query_id"] for row in generated_rows] == test_instance_ids
for row in generated_rows:
    assert set(row) == set(fieldnames)
    predicted_ids = [row[f"item_id_{rank}"] for rank in range(1, TOP_K + 1)]
    assert len(set(predicted_ids)) == TOP_K
    assert all(item_id in catalog_id_set for item_id in predicted_ids)

print(f"Saved {len(generated_rows):,} predictions to {SUBMISSION_PATH}")
print("Columns:", fieldnames)

Saved 19,858 predictions to 2022320319_final.csv
Columns: ['query_id', 'item_id_1', 'item_id_2', 'item_id_3', 'item_id_4', 'item_id_5', 'item_id_6', 'item_id_7', 'item_id_8', 'item_id_9', 'item_id_10']
